# 🎬 Override.AI — ViralCut Pro no Colab

Sobe o servidor com GPU gratuita do Colab e URL pública via ngrok.

**Execute as células em ordem. Tempo estimado: ~5 min.**

---

| Célula | O que faz |
|---|---|
| 1 | Instala dependências + ffmpeg |
| 2 | Copia arquivos do projeto |
| 3 | Configura chaves de API |
| 4 | Inicia ngrok + servidor |

In [ ]:
# ════════════════════════════════════════════════════════════
# CÉLULA 1 — Instala dependências
# ════════════════════════════════════════════════════════════
!pip install -q fastapi uvicorn python-dotenv python-multipart jinja2 aiofiles
!pip install -q google-genai google-generativeai
!pip install -q faster-whisper opencv-python-headless Pillow
!pip install -q yt-dlp pyngrok nest-asyncio requests python-telegram-bot
!apt-get install -y -q ffmpeg fonts-liberation
print('✅ Dependências instaladas!')

## 📁 Célula 2 — Arquivos do projeto

**Opção A (recomendada) — Google Drive:**
1. Faça upload da pasta `ViralCut_Pro` para o seu Google Drive
2. Ajuste `DRIVE_PATH` na célula abaixo para o caminho correto
3. Execute a célula

**Opção B — Git:**
Descomente a linha `!git clone ...` e comente/delete o bloco do Drive.

In [ ]:
# ════════════════════════════════════════════════════════════
# CÉLULA 2 — Copia arquivos do projeto
# ════════════════════════════════════════════════════════════
import os, shutil

PROJECT_DIR = '/content/viralcut'
for d in [PROJECT_DIR, PROJECT_DIR+'/templates', PROJECT_DIR+'/cortes',
          PROJECT_DIR+'/temp', PROJECT_DIR+'/temp_upload',
          PROJECT_DIR+'/logs', PROJECT_DIR+'/projetos']:
    os.makedirs(d, exist_ok=True)

# ── Opção A: Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ⚠️  Ajuste para o caminho da pasta no seu Drive:
DRIVE_PATH = '/content/drive/MyDrive/ViralCut_Pro'

for f in ['api.py', 'motor_novo.py', 'automacao_gemini.py', 'jarves.py', 'tasks.py']:
    src = DRIVE_PATH + '/' + f
    if os.path.exists(src):
        shutil.copy(src, PROJECT_DIR + '/' + f)
        print('✅', f)
    else:
        print('❌', f, '— não encontrado em', DRIVE_PATH)

for t in ['index.html', 'galeria.html']:
    src = DRIVE_PATH + '/templates/' + t
    if os.path.exists(src):
        shutil.copy(src, PROJECT_DIR + '/templates/' + t)
        print('✅ templates/' + t)

for font in ['arialbd.ttf', 'arial.ttf']:
    src = DRIVE_PATH + '/' + font
    if os.path.exists(src):
        shutil.copy(src, PROJECT_DIR + '/' + font)
        print('✅', font)

# ── Opção B: Git (descomente se preferir) ─────────────────────────────────────
# !git clone https://github.com/SEU_USUARIO/SEU_REPO.git /content/viralcut

# ── Fonte Linux (fallback automático caso arialbd.ttf não esteja no Drive) ────
_lf = '/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf'
_fd = PROJECT_DIR + '/arialbd.ttf'
if os.path.exists(_lf) and not os.path.exists(_fd):
    shutil.copy(_lf, _fd)
    print('✅ Fonte Linux copiada como arialbd.ttf')

os.chdir(PROJECT_DIR)
print('\nPronto! Diretório:', os.getcwd())

In [ ]:
# ════════════════════════════════════════════════════════════
# CÉLULA 3 — Configura chaves de API (.env)
# ════════════════════════════════════════════════════════════
from getpass import getpass
import os

os.chdir('/content/viralcut')
print('Cole cada chave abaixo (Enter para pular opcionais)\n')

GEMINI_KEY   = getpass('🔑 GEMINI_API_KEY  (obrigatório): ')
TELEGRAM_TOK = getpass('🤖 TELEGRAM_BOT_TOKEN  (opcional): ')
NGROK_TOKEN  = getpass('🌐 NGROK_AUTHTOKEN  (ngrok.com/dashboard): ')
API_TOKEN    = getpass('🔒 API_TOKEN  (senha do painel — crie qualquer uma): ')

with open('.env', 'w') as f:
    f.write('GEMINI_API_KEY='     + GEMINI_KEY   + '\n')
    f.write('TELEGRAM_BOT_TOKEN=' + TELEGRAM_TOK + '\n')
    f.write('NGROK_AUTHTOKEN='    + NGROK_TOKEN  + '\n')
    f.write('API_TOKEN='          + API_TOKEN    + '\n')

print('\n✅ .env criado!')

In [ ]:
# ════════════════════════════════════════════════════════════
# CÉLULA 4 — Inicia ngrok + servidor
# ════════════════════════════════════════════════════════════
import os, sys, time, threading
from pyngrok import ngrok
import nest_asyncio, uvicorn
from dotenv import load_dotenv

PROJECT_DIR = '/content/viralcut'
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
load_dotenv(override=True)

# ── ngrok ─────────────────────────────────────────────────────────────────────
ngrok_tok = os.getenv('NGROK_AUTHTOKEN', NGROK_TOKEN)
ngrok.set_auth_token(ngrok_tok)
ngrok.kill()   # mata túneis anteriores se existirem
tunnel     = ngrok.connect(8000)
PUBLIC_URL = tunnel.public_url

print('=' * 58)
print('  🌐 URL PÚBLICA :', PUBLIC_URL)
print('  🎬 Painel      :', PUBLIC_URL + '/')
print('  📡 API docs    :', PUBLIC_URL + '/docs')
print('=' * 58, '\n')

# ── Patch jarves.py: aponta API_BASE para ngrok em vez de localhost ───────────
with open('jarves.py', 'r', encoding='utf-8') as fh:
    code = fh.read()
lines = code.splitlines()
for i, ln in enumerate(lines):
    if ln.strip().startswith('API_BASE') and '=' in ln:
        lines[i] = 'API_BASE   = "' + PUBLIC_URL + '"'
        break
with open('jarves.py', 'w', encoding='utf-8') as fh:
    fh.write('\n'.join(lines))
print('✅ jarves.py → API_BASE =', PUBLIC_URL)

# ── Servidor ──────────────────────────────────────────────────────────────────
nest_asyncio.apply()
threading.Thread(
    target=lambda: uvicorn.run('api:app', host='0.0.0.0', port=8000, log_level='info'),
    daemon=True
).start()
time.sleep(3)
print('\n✅ Servidor no ar! Acesse a URL pública acima.')

## 📌 Referência rápida

| Ação | Como fazer |
|------|------------|
| Reabrir sessão | Execute **célula 4** novamente (nova URL ngrok) |
| Múltiplas chaves Gemini | Adicione `GEMINI_API_KEY_2`, `_3`... no `.env` |
| Ver logs em tempo real | Saída aparece abaixo da célula 4 |
| Parar servidor | `Runtime → Interrupt execution` |
| Usar GPU para Whisper | `Runtime → Change runtime type → T4 GPU` |

> ⚠️ A URL pública muda a cada sessão do Colab. Se o bot Jarves usa webhook, atualize com a nova URL.